In [1]:
import pandas as pd
import glob

files = glob.glob("../data/raw/*.csv")
df = pd.concat([pd.read_csv(f) for f in files], ignore_index=True)

df["start_time"] = pd.to_datetime(df["start_time"])
df["year"] = df["start_time"].dt.year

df = df[df["year"] >= 2018]

df.head()

,rdt_id,ns_lines,rdt_lines,rdt_lines_id,rdt_station_names,rdt_station_codes,cause_nl,cause_en,statistical_cause_nl,statistical_cause_en,cause_group,start_time,end_time,duration_minutes,year
2312,40500,Den Haag-Rotterdam; Leiden-Rotterdam,Den Haag HS - Rotterdam Centraal,11,"Delft,Delft Campus,Den Haag HS,Den Haag Moerwi...","DT, DTCP, GV, GVMW, RSW",brandmelding,fire alarm,brandmelding,fire alarm,external,2022-01-01 05:45:33,2022-01-01 06:34:58,49.0,2022
2313,40501,Amsterdam-Utrecht; Utrecht-Eindhoven,"'s-Hertogenbosch - Utrecht Centraal, Amsterdam...","136,151","Abcoude,Amsterdam Amstel,Amsterdam Bijlmer Are...","AC, ASA, ASB, ASD, ASDM, ASHD, BKL, CL, DVD, G...",herstelwerkzaamheden,repair works,herstelwerkzaamheden,repair works,engineering work,2022-01-01 06:23:54,2022-01-01 13:59:14,455.0,2022
2314,40502,Schiphol-Rotterdam (HSL),Rotterdam Centraal - Schiphol Airport (HSL),24,"Rotterdam Centraal,Schiphol Airport","RTD, SHL",brandmelding,fire alarm,brandmelding,fire alarm,external,2022-01-01 06:33:40,2022-01-01 06:34:27,1.0,2022
2315,40503,Eindhoven-Venlo,Eindhoven - Venlo,65,"Blerick,Deurne,Horst-Sevenum,Venlo","BR, DN, HRT, VL",aanrijding,collision,aanrijding,collision,accidents,2022-01-01 07:31:39,2022-01-01 11:26:38,235.0,2022
2316,40504,Alkmaar-Den Helder; Alkmaar-Hoorn,"Alkmaar - Den Helder, Alkmaar - Hoorn","162,163","Alkmaar Noord,Heerhugowaard","AMRN, HWD",aanrijding,collision,aanrijding,collision,accidents,2022-01-01 07:32:32,2022-01-01 07:42:25,10.0,2022


In [2]:
df["hour"] = df["start_time"].dt.hour
df["day_of_week"] = df["start_time"].dt.dayofweek
df["month"] = df["start_time"].dt.month


In [3]:
df["station_count"] = (
    df["rdt_station_codes"]
    .fillna("")
    .apply(lambda x: len(x.split(",")) if x else 0)
)


In [4]:
df["line_count"] = (
    df["rdt_lines_id"]
    .fillna("")
    .apply(lambda x: len(x.split(",")) if x else 0)
)


In [5]:
features = [
    "hour",
    "day_of_week",
    "month",
    "station_count",
    "line_count",
    "cause_group"
]

target = "duration_minutes"

df_model = df[features + [target]].dropna()

df_model.head()


,hour,day_of_week,month,station_count,line_count,cause_group,duration_minutes
2312,5,5,1,5,1,external,49.0
2313,6,5,1,19,2,engineering work,455.0
2314,6,5,1,2,1,external,1.0
2315,7,5,1,4,1,accidents,235.0
2316,7,5,1,2,2,accidents,10.0


In [6]:
df_model = pd.get_dummies(df_model, columns=["cause_group"], drop_first=True)


In [7]:
train = df_model[df["year"] < 2024]
test = df_model[df["year"] >= 2024]

X_train = train.drop(columns=[target])
y_train = train[target]

X_test = test.drop(columns=[target])
y_test = test[target]


/tmp/ipykernel_10540/4143546625.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  train = df_model[df["year"] < 2024]
/tmp/ipykernel_10540/4143546625.py:2: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  test = df_model[df["year"] >= 2024]


In [8]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error

model = RandomForestRegressor(
    n_estimators=100,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

preds = model.predict(X_test)
mae = mean_absolute_error(y_test, preds)

mae


183.0418313851499

### Baseline Model Result
The baseline Random Forest model achieved an MAE of 183.0418313851499 minutes on 2024–2025 data.


## Improved Model (Log-Transformed Target)
Applied log1p transformation to handle skewed disruption durations.


In [9]:
import numpy as np

y_train_log = np.log1p(y_train)
y_test_log = np.log1p(y_test)

model = RandomForestRegressor(
    n_estimators=200,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train_log)

preds_log = model.predict(X_test)
preds = np.expm1(preds_log)

mae_log = mean_absolute_error(y_test, preds)
mae_log


131.18474148454243

In [10]:
import pandas as pd

importances = pd.Series(
    model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)

importances.head(10)


hour                            0.215000
month                           0.193061
day_of_week                     0.145197
cause_group_rolling stock       0.138639
station_count                   0.123104
line_count                      0.061560
cause_group_external            0.050654
cause_group_staff               0.035100
cause_group_infrastructure      0.014604
cause_group_engineering work    0.008446
dtype: float64

### Feature Importance
The most influential features were number of affected stations, cause group, and time-related variables.


In [11]:
import joblib

joblib.dump(model, "../models/disruption_duration_model.pkl")


['../models/disruption_duration_model.pkl']